# Project :- Vehicle Health & Maintenance Analysis System

### Data Generation

In [1]:
import pandas as pd
import numpy as np
import random
import os
from faker import Faker

In [2]:
fake = Faker('en_IN')
np.random.seed(42)

In [4]:
vehicles = pd.DataFrame({
    'vehicle_id':   [f'VH{i:03d}' for i in range(1, 501)],
    'make':         np.random.choice(['Toyota','Tata','Honda','Hyundai','Maruti','Ford','Mahindra'], 500),
    'model_year':   np.random.randint(2015, 2024, 500),
    'fuel_type':    np.random.choice(['Petrol','Diesel','CNG','Electric'], 500, p=[0.4,0.35,0.15,0.10]),
    'odometer_km':  np.round(np.random.uniform(5000, 200000, 500), 1),
    'city':         np.random.choice(['Hyderabad','Mumbai','Delhi','Bengaluru','Chennai','Pune','Kolkata'], 500),
})

In [7]:
drivers = pd.DataFrame({
    'driver_id':    [f'DR{i:03d}' for i in range(1, 201)],
    'driver_name':  [fake.name() for _ in range(200)],
    'vehicle_id':   np.random.choice(vehicles['vehicle_id'], 200),
    'city':         np.random.choice(['Hyderabad','Mumbai','Delhi','Bengaluru','Chennai'], 200),
    'experience':   np.random.randint(1, 20, 200),
})

In [9]:
n = 6000
maintenance = pd.DataFrame({
    'log_id':        range(1, n+1),
    'vehicle_id':    np.random.choice(vehicles['vehicle_id'], n),
    'driver_id':     np.random.choice(drivers['driver_id'],   n),
    'service_date':  pd.date_range('2021-01-01', periods=n, freq='6h').strftime('%Y-%m-%d'),
    'service_type':  np.random.choice(['Oil Change','Brake Check','Full Service',
                                        'Tyre Rotation','Battery Replacement',
                                        'AC Service','Engine Tune-up','Wheel Alignment'], n),
    'cost_inr':      np.round(np.random.uniform(500, 20000, n), 2),
    'km_at_service': np.round(np.random.uniform(5000, 195000, n), 1),
    'overdue_flag':  np.random.choice([0, 1], n, p=[0.7, 0.3]),
    'vendor':        np.random.choice(['AutoCare Hub','SpeedWheels','QuickFix Motors','PrimeTech'], n),
})

In [11]:
n = 7000
fuel_logs = pd.DataFrame({
    'fuel_id':    range(1, n+1),
    'vehicle_id': np.random.choice(vehicles['vehicle_id'], n),
    'fill_date':  pd.date_range('2021-01-01', periods=n, freq='5h').strftime('%Y-%m-%d'),
    'fuel_type':  np.random.choice(['Petrol','Diesel','CNG'], n, p=[0.45,0.40,0.15]),
    'litres':     np.round(np.random.uniform(15, 60, n), 2),
    'km_driven':  np.round(np.random.uniform(200, 700, n), 1),
    'cost_inr':   np.round(np.random.uniform(1500, 6000, n), 2),
    'station':    np.random.choice(['Indian Oil','HP','BPCL','Shell','Reliance'], n),
})
fuel_logs['km_per_litre'] = (fuel_logs['km_driven'] / fuel_logs['litres']).round(2)

In [13]:
n = 1300
alerts = pd.DataFrame({
    'alert_id':   range(1, n+1),
    'vehicle_id': np.random.choice(vehicles['vehicle_id'], n),
    'alert_date': pd.date_range('2021-01-01', periods=n, freq='12h').strftime('%Y-%m-%d'),
    'alert_type': np.random.choice(['Engine Warning','Battery Low','Brake Wear',
                                     'Oil Level Low','Overheating','ABS Fault'], n),
    'severity':   np.random.choice(['HIGH','MEDIUM','LOW'], n, p=[0.3,0.4,0.3]),
    'resolved':   np.random.choice([0, 1], n, p=[0.35, 0.65]),
    'fault_code': np.random.choice(['P0300','P0420','B0020','C0031','P0217','N/A'], n),
})

In [15]:
os.makedirs('data/raw', exist_ok=True)
 
vehicles.to_csv('data/raw/vehicles.csv',       index=False)
drivers.to_csv('data/raw/drivers.csv',         index=False)
maintenance.to_csv('data/raw/maintenance.csv', index=False)
fuel_logs.to_csv('data/raw/fuel_logs.csv',     index=False)
alerts.to_csv('data/raw/alerts.csv',           index=False)
 
print("Data generated!")
total = 0
for name, df in zip(['Vehicles','Drivers','Maintenance','Fuel Logs','Alerts'],
                    [vehicles, drivers, maintenance, fuel_logs, alerts]):
    print(f"   {name:<15} -> {len(df):>6,} rows")
    total += len(df)
print(f"   {'TOTAL':<15} -> {total:>6,} rows")

Data generated!
   Vehicles        ->    500 rows
   Drivers         ->    200 rows
   Maintenance     ->  6,000 rows
   Fuel Logs       ->  7,000 rows
   Alerts          ->  1,300 rows
   TOTAL           -> 15,000 rows


### Data Cleaning

In [18]:
os.makedirs('data/clean', exist_ok=True)

In [20]:
df = pd.read_csv('data/raw/vehicles.csv')
 
# Injecting nulls for practice
df.loc[df.sample(frac=0.05).index, 'odometer_km'] = np.nan
df.loc[df.sample(frac=0.04).index, 'fuel_type']   = np.nan
df.loc[df.sample(frac=0.04).index, 'city']         = np.nan
 
# Clean
df.drop_duplicates(inplace=True)
df['fuel_type']   = df['fuel_type'].str.strip().str.title().fillna('Petrol')
df['city']        = df['city'].str.strip().str.title().fillna('Hyderabad')
df['odometer_km'] = pd.to_numeric(df['odometer_km'], errors='coerce') \
                      .fillna(df['odometer_km'].median()).round(1)
 
# Feature engineering
df['vehicle_age'] = 2024 - df['model_year']
df['age_group']   = pd.cut(df['vehicle_age'],
                            bins=[0,3,6,10,20],
                            labels=['New(0-3)','Mid(3-6)','Old(6-10)','VeryOld(10+)'])
 
df.to_csv('D:/Data Analytics/vehicles.csv', index=False)
print(f"vehicles    : {len(df)} rows | nulls: {df.isnull().sum().sum()}")

vehicles    : 500 rows | nulls: 0


In [22]:
df = pd.read_csv('data/raw/drivers.csv')
 
df.loc[df.sample(frac=0.05).index, 'experience'] = np.nan
 
df.drop_duplicates(inplace=True)
df['driver_name'] = df['driver_name'].str.strip().str.title()
df['city']        = df['city'].str.strip().str.title()
df['experience']  = pd.to_numeric(df['experience'], errors='coerce') \
                      .fillna(df['experience'].median()).astype(int)
 
df.to_csv('D:/Data Analytics/drivers.csv', index=False)
print(f"drivers     : {len(df)} rows | nulls: {df.isnull().sum().sum()}")

drivers     : 200 rows | nulls: 0


In [24]:
df = pd.read_csv('data/raw/maintenance.csv')
 
df.loc[df.sample(frac=0.06).index, 'cost_inr']    = np.nan
df.loc[df.sample(frac=0.04).index, 'vendor']       = np.nan
df.loc[df.sample(frac=0.03).index, 'service_date'] = np.nan
 
df.drop_duplicates(inplace=True)
df.dropna(subset=['service_date'], inplace=True)
df['service_date'] = pd.to_datetime(df['service_date'], errors='coerce')
df.dropna(subset=['service_date'], inplace=True)
 
df['cost_inr']     = pd.to_numeric(df['cost_inr'], errors='coerce') \
                       .fillna(df['cost_inr'].median()).round(2)
df['vendor']       = df['vendor'].fillna('Unknown Vendor')
df['service_type'] = df['service_type'].str.strip().str.title()
 
# Cap cost outliers at 99th percentile
cap = df['cost_inr'].quantile(0.99)
df['cost_inr'] = df['cost_inr'].clip(upper=cap)
 
# Feature engineering
df['service_year']  = df['service_date'].dt.year
df['service_month'] = df['service_date'].dt.month
df['month_name']    = df['service_date'].dt.strftime('%b')
df['quarter']       = df['service_date'].dt.quarter.map(
                        {1:'Q1',2:'Q2',3:'Q3',4:'Q4'})
 
df.to_csv('D:/Data Analytics/maintenance.csv', index=False)
print(f"maintenance : {len(df)} rows | nulls: {df.isnull().sum().sum()}")

maintenance : 5820 rows | nulls: 0


In [26]:
df = pd.read_csv('data/raw/fuel_logs.csv')
 
df.loc[df.sample(frac=0.05).index, 'litres']    = np.nan
df.loc[df.sample(frac=0.04).index, 'km_driven'] = np.nan
 
df.drop_duplicates(inplace=True)
df['fill_date'] = pd.to_datetime(df['fill_date'], errors='coerce')
df.dropna(subset=['fill_date'], inplace=True)
 
df['litres']    = pd.to_numeric(df['litres'], errors='coerce') \
                    .fillna(df['litres'].median()).round(2)
df['km_driven'] = pd.to_numeric(df['km_driven'], errors='coerce') \
                    .fillna(df['km_driven'].median()).round(1)
 
# Recalculate km_per_litre cleanly after imputation
df['km_per_litre'] = (df['km_driven'] / df['litres']).round(2)
 
# Remove outliers: km_per_litre outside 5–35
df = df[df['km_per_litre'].between(5, 35)]
 
# Feature engineering
df['fill_year']  = df['fill_date'].dt.year
df['fill_month'] = df['fill_date'].dt.month
df['cost_per_km'] = (df['cost_inr'] / df['km_driven']).round(2)

df.to_csv('D:/Data Analytics/fuel_logs.csv', index=False)
print(f"fuel_logs   : {len(df)} rows | nulls: {df.isnull().sum().sum()}")

fuel_logs   : 6614 rows | nulls: 0


In [28]:
df = pd.read_csv('data/raw/alerts.csv')
 
df.loc[df.sample(frac=0.05).index, 'fault_code'] = np.nan
 
df.drop_duplicates(inplace=True)
df['alert_date'] = pd.to_datetime(df['alert_date'], errors='coerce')
df.dropna(subset=['alert_date'], inplace=True)
 
df['severity']   = df['severity'].str.strip().str.upper()
df['alert_type'] = df['alert_type'].str.strip().str.title()
df['fault_code'] = df['fault_code'].fillna('UNKNOWN')
 
# Keep only valid severity values
df = df[df['severity'].isin(['HIGH','MEDIUM','LOW'])]
 
# Feature engineering
df['alert_year']  = df['alert_date'].dt.year
df['alert_month'] = df['alert_date'].dt.month
df['status']      = df['resolved'].map({1:'Resolved', 0:'Open'})
 
df.to_csv('D:/Data Analytics/alerts.csv', index=False)
print(f"alerts      : {len(df)} rows | nulls: {df.isnull().sum().sum()}")
 
print("\nCleaning complete -> data/clean/")

alerts      : 1300 rows | nulls: 0

Cleaning complete -> data/clean/
